In [30]:
import run_noise_only
from matplotlib import pyplot as plt
import h5py
import numpy as np

In [31]:
import lisatools

In [45]:
from lisatools.sensitivity import (
    CompositeSensitivityMatrix,
    UnequalArmInstrumentNoise,
    LinkDelayTable,
    UNEQUAL_ARM_LINKS
)
from lisatools.stochastic import HyperbolicTangentGalacticForeground
from lisatools.domains import WDMSettings

In [46]:
with h5py.File("../../../NOISE_731d_2.5s_L1_source0_0_20251206T220508924302Z.h5", "r") as f:
    #print(f['ltts'].keys())
    ltts = {link: np.array(f[f'ltts/ltt_{link}']) for link in UNEQUAL_ARM_LINKS}
    ltt_t0 = f['ltts/sampling'].attrs['t0']
    ltt_dt = f['ltts/sampling'].attrs['dt']
    ltt_t = ltt_t0 + ltt_dt * np.arange(ltts[12].shape[0])
    

In [ ]:
# 12, 23, 31, 13, 32, 21

In [51]:
ltts_arr = np.stack([ltts[link] for link in UNEQUAL_ARM_LINKS]).T

In [41]:
LinkDelayTable?

Init signature: LinkDelayTable(t, ltts, data_t0: 'float' = 0.0)
Docstring:     
Per-link light travel times tabulated against an absolute clock.

The delays breathe by ~1.5-1.8% over a two-year run -- far more than the
~0.2% spread between the time-averaged arms -- so collapsing them to one
epoch throws away the dominant variation. This holds the delays as a time
series and averages them **over each WDM time slice**, giving
:class:`UnequalArmInstrumentNoise` one delay set per wavelet time column
that represents the whole column rather than a point sample of it.

Holds plain numpy arrays only, so it rides through the settings-tree
deepcopy / pickle round trip (sprint deepcopy/pickle rule).

Args:
    t: Sample times in seconds on the **absolute** clock the delays are
        tabulated against (for a mojito L1 brick, ``/ltts/sampling`` ``t0``
        plus ``k*dt``). Must be increasing.
    ltts: ``(len(t), 6)`` light travel times in :data:`UNEQUAL_ARM_LINKS`
        order.
    data_t0: A

In [34]:
CompositeSensitivityMatrix?

Init signature:
CompositeSensitivityMatrix(
    settings: 'domains.DomainSettingsBase',
    components: 'Sequence[NoiseComponent]',
    skip_inv_det: 'bool' = False,
)
Docstring:     
Sensitivity matrix built as a sum of :class:`NoiseComponent` objects.

Args:
    settings: Domain settings the matrix is evaluated on (FD, WDM, …).
    components: :class:`NoiseComponent` list to sum. All must produce the same
        ``(nch, nch, *basis_shape_active)`` shape.
    skip_inv_det: Skip determinant/inverse computation (e.g. for slicing).
File:           ~/code/lisa_sprint_2026_clean/LISAanalysistools/src/lisatools/sensitivity.py
Type:           type
Subclasses:     MojitoNoiseSensitivityMatrix

In [35]:
HyperbolicTangentGalacticForeground?

Init signature: HyperbolicTangentGalacticForeground()
Docstring:      Hyperbolic Tangent-based foreground fitting function.
File:           ~/code/lisa_sprint_2026_clean/LISAanalysistools/src/lisatools/stochastic.py
Type:           ABCMeta
Subclasses:     FittedHyperbolicTangentGalacticForeground

In [36]:
wdm = WDMSettings(Nf=768, Nt=1024, dt=5.0)
wdm.Tobs / 3600 / 24

45.51111111111111

In [52]:
ldt = LinkDelayTable(ltt_t, ltts_arr, ltt_t0)

In [ ]:
ldt

In [53]:
CompositeSensitivityMatrix(wdm, [
    UnequalArmInstrumentNoise(ldt),
    HyperbolicTangentGalacticForeground()
])

KeyboardInterrupt: 

In [ ]:
with h5py.File("./noise-galfor-pe/noise_foreground_full5_testing.h5", 'r') as f:
    #f['global_fit/chain/galfor']
    # need to make noise wpsd from params, and galfor psd
    
